# 00 — Learning $\kappa$

A minimal example with a known density ratio. We fit FSNM and evaluate its estimate of $\kappa$.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import ParameterGrid

from fsnm import empirical_loss, fit_fsnm


def basis(t):
    """Unit-norm, zero-mean function in L²(Unif[-1, 1])."""
    return np.sqrt(2) * np.cos(np.pi * np.asarray(t))


def kappa_exact(x_values, y_values):
    return 1 + SIGMA * np.outer(basis(x_values), basis(y_values))


def sample_joint(size, seed):
    rng = np.random.default_rng(seed)
    upper_bound = 1 + 2 * SIGMA
    x_parts = []
    y_parts = []
    n_accepted = 0

    while n_accepted < size:
        x = rng.uniform(-1, 1, size)
        y = rng.uniform(-1, 1, size)
        density_ratio = 1 + SIGMA * basis(x) * basis(y)
        accepted = rng.uniform(size=size) < density_ratio / upper_bound
        x_parts.append(x[accepted])
        y_parts.append(y[accepted])
        n_accepted += accepted.sum()

    x = np.concatenate(x_parts)[:size]
    y = np.concatenate(y_parts)[:size]
    return x, y

We use

$$\kappa(x,y)=1+\sigma e(x)e(y),\qquad e(t)=\sqrt{2}\cos(\pi t),$$

with $\sigma=0.4$. Since $|e(x)e(y)|\leq 2$, we have $\kappa(x,y)\geq 1-2\sigma=0.2$.

In [2]:
SIGMA = 0.4
N_TRAIN = 5_000
N_VALIDATION = 2_000
SEED = 0

x_train, y_train = sample_joint(N_TRAIN, seed=12)
x_validation, y_validation = sample_joint(N_VALIDATION, seed=99)

In [3]:
parameter_grid = {
    "n_iterations": [5, 10, 20, 40],
    "step_size": [0.1, 0.2, 0.5],
    "max_depth": [3, None],
    "min_samples_leaf": [100, 300, 600],
}

grid_results = []
best_validation_loss = np.inf

for parameters in ParameterGrid(parameter_grid):
    candidate_phi, candidate_psi, candidate_values, _ = fit_fsnm(
        x_train,
        y_train,
        rank=1,
        seed=SEED,
        **parameters,
    )
    singular_scale = np.sqrt(candidate_values)
    phi_validation = candidate_phi.predict(x_validation[:, None]) * singular_scale
    psi_validation = candidate_psi.predict(y_validation[:, None]) * singular_scale
    validation_loss = float(empirical_loss(phi_validation, psi_validation))
    grid_results.append(
        {
            **parameters,
            "validation_loss": validation_loss,
            "singular_value": float(candidate_values[0]),
        }
    )

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        phi = candidate_phi
        psi = candidate_psi
        singular_values = candidate_values

grid_results = sorted(grid_results, key=lambda result: result["validation_loss"])
best_parameters = {
    name: grid_results[0][name]
    for name in parameter_grid
}

print(f"FSNM evaluated combinations: {len(grid_results)}")
print(f"FSNM best hyperparameters: {best_parameters}")
print(f"FSNM validation loss: {best_validation_loss:.4f}")
print(f"true singular value:    {SIGMA:.4f}")
print(f"FSNM singular value:     {singular_values[0]:.4f}")
print("\nFSNM top five configurations:")
for result in grid_results[:5]:
    print(result)

FSNM evaluated combinations: 72
FSNM best hyperparameters: {'n_iterations': 20, 'step_size': 0.1, 'max_depth': None, 'min_samples_leaf': 300}
FSNM validation loss: -0.1517
true singular value:    0.4000
FSNM singular value:     0.3986

FSNM top five configurations:
{'max_depth': None, 'min_samples_leaf': 300, 'n_iterations': 20, 'step_size': 0.1, 'validation_loss': -0.15165725579759975, 'singular_value': 0.39858598771038317}
{'max_depth': None, 'min_samples_leaf': 300, 'n_iterations': 40, 'step_size': 0.1, 'validation_loss': -0.1498480681089942, 'singular_value': 0.4092386365536989}
{'max_depth': None, 'min_samples_leaf': 300, 'n_iterations': 10, 'step_size': 0.2, 'validation_loss': -0.14966303166995312, 'singular_value': 0.39911339255926}
{'max_depth': None, 'min_samples_leaf': 300, 'n_iterations': 20, 'step_size': 0.2, 'validation_loss': -0.14943712293756334, 'singular_value': 0.40918534936789136}
{'max_depth': None, 'min_samples_leaf': 300, 'n_iterations': 10, 'step_size': 0.5, 'val

In [4]:
grid = np.linspace(-1, 1, 160)
kappa_true = kappa_exact(grid, grid)
kappa_fsnm = 1 + (
    phi.predict(grid[:, None]) * singular_values
) @ psi.predict(grid[:, None]).T
error_fsnm = kappa_fsnm - kappa_true
rmse_fsnm = np.sqrt(np.mean(error_fsnm**2))

value_limits = (
    min(kappa_true.min(), kappa_fsnm.min()),
    max(kappa_true.max(), kappa_fsnm.max()),
)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)
value_panels = [
    (kappa_true, r"true $\kappa$"),
    (kappa_fsnm, r"FSNM $\widehat\kappa$"),
]
for axis, (values, title) in zip(axes[:2], value_panels):
    image = axis.imshow(
        values.T,
        origin="lower",
        extent=[-1, 1, -1, 1],
        cmap="coolwarm",
        vmin=value_limits[0],
        vmax=value_limits[1],
    )
    axis.set(title=title, xlabel="$x$", ylabel="$y$")
    fig.colorbar(image, ax=axis, shrink=0.82)

error_limit = np.max(np.abs(error_fsnm))
image = axes[2].imshow(
    error_fsnm.T,
    origin="lower",
    extent=[-1, 1, -1, 1],
    cmap="coolwarm",
    vmin=-error_limit,
    vmax=error_limit,
)
axes[2].set(title=r"$\widehat\kappa-\kappa$", xlabel="$x$", ylabel="$y$")
fig.colorbar(image, ax=axes[2], shrink=0.82)

project_directory = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
figure_directory = project_directory / "figures"
figure_directory.mkdir(exist_ok=True)
figure_path = figure_directory / "00_kappa_comparison.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
print(f"FSNM grid RMSE: {rmse_fsnm:.4f}")
print(f"figure saved to: {figure_path}")
plt.show()

FSNM grid RMSE: 0.1072
figure saved to: /home/thiago/Projects/paper-fsnm/code/fsnm/figures/00_kappa_comparison.png


/tmp/ipykernel_47933/2394269355.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
